In [4]:
import pandas as pd
df = pd.read_csv(r"W:\Learn_Program\project_bigdata\Part_2\data\data.csv")

In [5]:
# ============================================================
# 2.4 Text Preprocessing & Feature Engineering
# ============================================================

import re
import pandas as pd
import numpy as np

print("Data shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nตัวอย่างข้อมูล:")
display(
    df[["text", "rating", "helpful_vote"]].head()
)

Data shape: (30000, 27)

Columns:
['Unnamed: 0', 'rating', 'title_review', 'text', 'images_review', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'main_category', 'title_product', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images_product', 'videos', 'store', 'categories', 'details', 'bought_together', 'subtitle', 'author', '_merge']

ตัวอย่างข้อมูล:


,text,rating,helpful_vote
0,"Not super comfy, but I guess can't expect much...",3.0,0
1,"I should have purchased this years ago, Final...",5.0,1
2,Jack Russell Terriers are one of our favorite ...,5.0,0
3,This is the perfect set of pens. The colors ar...,5.0,0
4,Works fine. No issues.,5.0,0


In [6]:
# ============================================================
# ตรวจสอบข้อความรีวิว
# ============================================================

print("จำนวนข้อมูลทั้งหมด:", len(df))

print(
    "จำนวน text ที่เป็น missing:",
    df["text"].isna().sum()
)

print(
    "จำนวน text ที่เป็นค่าว่าง:",
    (df["text"].fillna("").str.strip() == "").sum()
)

print("\nตัวอย่างข้อความรีวิว:")
display(
    df[["text", "rating"]].head(10)
)

จำนวนข้อมูลทั้งหมด: 30000
จำนวน text ที่เป็น missing: 8
จำนวน text ที่เป็นค่าว่าง: 38

ตัวอย่างข้อความรีวิว:


,text,rating
0,"Not super comfy, but I guess can't expect much...",3.0
1,"I should have purchased this years ago, Final...",5.0
2,Jack Russell Terriers are one of our favorite ...,5.0
3,This is the perfect set of pens. The colors ar...,5.0
4,Works fine. No issues.,5.0
5,The picture was clear and precise,5.0
6,"I unpacked the printer very carefully, removed...",1.0
7,Works Great easy to use with my phone .,5.0
8,Tabbies are a life saver for file folders. I c...,5.0
9,All our office phones are these guys. Function...,5.0


In [7]:
# ============================================================
# Text Preprocessing
# ============================================================

def clean_text(text):
    """
    ทำความสะอาดข้อความรีวิว
    """

    # กรณี text เป็น missing
    if pd.isna(text):
        return ""

    # แปลงเป็น string
    text = str(text)

    # แปลงตัวอักษรเป็น lowercase
    text = text.lower()

    # ลบ HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # ลบ URL
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # ลบช่องว่างซ้ำ
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# สร้าง column ใหม่
df["clean_text"] = df["text"].apply(clean_text)


# ตรวจสอบผลลัพธ์
display(
    df[["text", "clean_text"]].head(10)
)

,text,clean_text
0,"Not super comfy, but I guess can't expect much...","not super comfy, but i guess can't expect much..."
1,"I should have purchased this years ago, Final...","i should have purchased this years ago, finall..."
2,Jack Russell Terriers are one of our favorite ...,jack russell terriers are one of our favorite ...
3,This is the perfect set of pens. The colors ar...,this is the perfect set of pens. the colors ar...
4,Works fine. No issues.,works fine. no issues.
5,The picture was clear and precise,the picture was clear and precise
6,"I unpacked the printer very carefully, removed...","i unpacked the printer very carefully, removed..."
7,Works Great easy to use with my phone .,works great easy to use with my phone .
8,Tabbies are a life saver for file folders. I c...,tabbies are a life saver for file folders. i c...
9,All our office phones are these guys. Function...,all our office phones are these guys. function...


In [8]:
# ============================================================
# Feature 1: Sentiment Score
# ============================================================

import nltk

from nltk.sentiment import SentimentIntensityAnalyzer

# ดาวน์โหลด VADER Lexicon
nltk.download(
    "vader_lexicon",
    quiet=True
)

# สร้าง Sentiment Analyzer
sia = SentimentIntensityAnalyzer()

print("VADER Sentiment Analyzer พร้อมใช้งาน")

VADER Sentiment Analyzer พร้อมใช้งาน


In [9]:
# ============================================================
# สร้าง Sentiment Score
# ============================================================

df["sentiment_score"] = df["clean_text"].apply(
    lambda x: sia.polarity_scores(x)["compound"]
)

# แสดงผล
display(
    df[
        [
            "text",
            "rating",
            "sentiment_score"
        ]
    ].head(10)
)

,text,rating,sentiment_score
0,"Not super comfy, but I guess can't expect much...",3.0,-0.9268
1,"I should have purchased this years ago, Final...",5.0,0.8047
2,Jack Russell Terriers are one of our favorite ...,5.0,0.8434
3,This is the perfect set of pens. The colors ar...,5.0,0.9201
4,Works fine. No issues.,5.0,-0.1027
5,The picture was clear and precise,5.0,0.3818
6,"I unpacked the printer very carefully, removed...",1.0,-0.4723
7,Works Great easy to use with my phone .,5.0,0.7906
8,Tabbies are a life saver for file folders. I c...,5.0,0.5927
9,All our office phones are these guys. Function...,5.0,0.6369


In [10]:
# ============================================================
# แปลง Sentiment Score เป็นกลุ่ม
# ============================================================

def sentiment_label(score):

    if score >= 0.05:
        return "Positive"

    elif score <= -0.05:
        return "Negative"

    else:
        return "Neutral"


df["sentiment"] = df["sentiment_score"].apply(
    sentiment_label
)


# ตรวจสอบจำนวนแต่ละกลุ่ม
print("จำนวนรีวิวในแต่ละ Sentiment:")

display(
    df["sentiment"].value_counts()
)

จำนวนรีวิวในแต่ละ Sentiment:


sentiment
Positive    22732
Negative     3771
Neutral      3497
Name: count, dtype: int64

In [11]:
# ============================================================
# ตรวจสอบ Sentiment กับ Rating
# ============================================================

display(
    df[
        [
            "rating",
            "sentiment_score",
            "sentiment"
        ]
    ].head(20)
)

print("\nค่าเฉลี่ย Sentiment Score ตาม Rating:")

display(
    df.groupby("rating")["sentiment_score"]
      .mean()
      .reset_index()
)

,rating,sentiment_score,sentiment
0,3.0,-0.9268,Negative
1,5.0,0.8047,Positive
2,5.0,0.8434,Positive
3,5.0,0.9201,Positive
4,5.0,-0.1027,Negative
5,5.0,0.3818,Positive
6,1.0,-0.4723,Negative
7,5.0,0.7906,Positive
8,5.0,0.5927,Positive
9,5.0,0.6369,Positive



ค่าเฉลี่ย Sentiment Score ตาม Rating:


,rating,sentiment_score
0,1.0,-0.150062
1,2.0,0.026208
2,3.0,0.252341
3,4.0,0.478214
4,5.0,0.598103


In [12]:
# ============================================================
# Feature 2: Aspect Indicators
# ============================================================

aspects = {

    # ราคา
    "price": [
        "price",
        "expensive",
        "cheap",
        "cost",
        "value"
    ],

    # คุณภาพ
    "quality": [
        "quality",
        "good quality",
        "poor quality",
        "durable",
        "durability"
    ],

    # การจัดส่ง
    "delivery": [
        "delivery",
        "shipping",
        "shipped",
        "arrived",
        "package",
        "packaging"
    ],

    # ตัวสินค้า
    "product": [
        "product",
        "item",
        "works",
        "working",
        "broken",
        "defective"
    ]
}


print("Aspect ที่ใช้วิเคราะห์:")
for aspect in aspects:
    print("-", aspect)

Aspect ที่ใช้วิเคราะห์:
- price
- quality
- delivery
- product


In [13]:
# ============================================================
# สร้าง Aspect Indicator
# ============================================================

for aspect, keywords in aspects.items():

    # สร้าง regex pattern
    pattern = "|".join(
        re.escape(keyword)
        for keyword in keywords
    )

    # ตรวจสอบว่าข้อความมี keyword หรือไม่
    df[f"aspect_{aspect}"] = (
        df["clean_text"]
        .str.contains(
            pattern,
            regex=True,
            na=False
        )
        .astype(int)
    )


print("สร้าง Aspect Indicators เรียบร้อยแล้ว")

สร้าง Aspect Indicators เรียบร้อยแล้ว


In [14]:
# ============================================================
# ตรวจสอบ Aspect Indicators
# ============================================================

aspect_columns = [
    "aspect_price",
    "aspect_quality",
    "aspect_delivery",
    "aspect_product"
]

display(
    df[
        [
            "text",
            "rating",
            "sentiment_score"
        ] + aspect_columns
    ].head(20)
)

,text,rating,sentiment_score,aspect_price,aspect_quality,aspect_delivery,aspect_product
0,"Not super comfy, but I guess can't expect much...",3.0,-0.9268,1,0,0,0
1,"I should have purchased this years ago, Final...",5.0,0.8047,1,0,0,0
2,Jack Russell Terriers are one of our favorite ...,5.0,0.8434,0,0,0,0
3,This is the perfect set of pens. The colors ar...,5.0,0.9201,0,0,0,0
4,Works fine. No issues.,5.0,-0.1027,0,0,0,1
5,The picture was clear and precise,5.0,0.3818,0,0,0,0
6,"I unpacked the printer very carefully, removed...",1.0,-0.4723,0,0,0,0
7,Works Great easy to use with my phone .,5.0,0.7906,0,0,0,1
8,Tabbies are a life saver for file folders. I c...,5.0,0.5927,0,0,0,0
9,All our office phones are these guys. Function...,5.0,0.6369,0,0,0,0


In [15]:
# ============================================================
# จำนวนรีวิวที่พูดถึงแต่ละ Aspect
# ============================================================

aspect_counts = (
    df[aspect_columns]
    .sum()
    .sort_values(
        ascending=False
    )
)

print("จำนวนรีวิวที่กล่าวถึงแต่ละ Aspect:")

display(
    aspect_counts
)

จำนวนรีวิวที่กล่าวถึงแต่ละ Aspect:


aspect_product     6567
aspect_price       4316
aspect_quality     3374
aspect_delivery    1894
dtype: int64

In [16]:
# ============================================================
# ค่าเฉลี่ย Rating ของรีวิวที่พูดถึงแต่ละ Aspect
# ============================================================

aspect_rating = {}

for aspect in aspect_columns:

    # เลือกเฉพาะ review ที่มี aspect = 1
    subset = df[df[aspect] == 1]

    # คำนวณค่าเฉลี่ย rating
    aspect_rating[aspect] = {
        "review_count": len(subset),
        "avg_rating": subset["rating"].mean(),
        "avg_sentiment": subset["sentiment_score"].mean()
    }


aspect_rating_df = (
    pd.DataFrame(aspect_rating)
    .T
    .reset_index()
    .rename(
        columns={
            "index": "aspect"
        }
    )
)

display(
    aspect_rating_df
)

,aspect,review_count,avg_rating,avg_sentiment
0,aspect_price,4316.0,4.201112,0.552363
1,aspect_quality,3374.0,4.318613,0.576181
2,aspect_delivery,1894.0,4.095037,0.464849
3,aspect_product,6567.0,4.171311,0.486721


In [17]:
# ============================================================
# สรุป Text Features
# ============================================================

text_features = [
    "sentiment_score",
    "sentiment",
    "aspect_price",
    "aspect_quality",
    "aspect_delivery",
    "aspect_product"
]

print("Text Features ที่สร้าง:")
for feature in text_features:
    print("-", feature)

print("\nจำนวน Text Features:", len(text_features))

Text Features ที่สร้าง:
- sentiment_score
- sentiment
- aspect_price
- aspect_quality
- aspect_delivery
- aspect_product

จำนวน Text Features: 6


In [18]:
# ============================================================
# Final Check
# ============================================================

display(
    df[
        [
            "text",
            "rating",
            "helpful_vote",
            "sentiment_score",
            "sentiment",
            "aspect_price",
            "aspect_quality",
            "aspect_delivery",
            "aspect_product"
        ]
    ].head(10)
)

,text,rating,helpful_vote,sentiment_score,sentiment,aspect_price,aspect_quality,aspect_delivery,aspect_product
0,"Not super comfy, but I guess can't expect much...",3.0,0,-0.9268,Negative,1,0,0,0
1,"I should have purchased this years ago, Final...",5.0,1,0.8047,Positive,1,0,0,0
2,Jack Russell Terriers are one of our favorite ...,5.0,0,0.8434,Positive,0,0,0,0
3,This is the perfect set of pens. The colors ar...,5.0,0,0.9201,Positive,0,0,0,0
4,Works fine. No issues.,5.0,0,-0.1027,Negative,0,0,0,1
5,The picture was clear and precise,5.0,0,0.3818,Positive,0,0,0,0
6,"I unpacked the printer very carefully, removed...",1.0,11,-0.4723,Negative,0,0,0,0
7,Works Great easy to use with my phone .,5.0,0,0.7906,Positive,0,0,0,1
8,Tabbies are a life saver for file folders. I c...,5.0,0,0.5927,Positive,0,0,0,0
9,All our office phones are these guys. Function...,5.0,0,0.6369,Positive,0,0,0,0
